# Our own Hinglish BERT — continued MLM pre-training of mBERT
T4 GPU. Resumable (checkpoints to Drive every 1000 steps; re-run the same cell after a disconnect).

| Model | What | Time (T4) |
|---|---|---|
| `ourbert-tapt` | mBERT + MLM on SentiMix **train** tweets only (~12k) | ~5 min |
| `ourbert` | mBERT + MLM on train tweets + L3Cube-HingLID (~56k tweets) | ~15–20 min |
| `hingmbert` | L3Cube's mBERT pre-trained on full HingCorpus (52.9M) — reference, no pre-training needed | 0 |

Then each is fine-tuned with the **same** sweep + 3 seeds as every other model (~25 min each).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/Raajp10/hinglish-sentimix.git'
import os
WORK = '/content/drive/MyDrive/hinglish-sentimix'
if not os.path.exists(WORK):
    !git clone $REPO_URL $WORK
else:
    !cd $WORK && git pull
%cd $WORK
try:  # optional: HF token from Colab secrets (avoids rate limits)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
!pip install -q -r requirements.txt
!python -m src.prepare > /dev/null && python -m src.normalize | head -1

## 1. Pre-train (MLM)

In [ ]:
!python -m src.mlm_pretrain --name ourbert-tapt --sources sentimix --epochs 3

In [ ]:
!python -m src.mlm_pretrain --name ourbert --sources sentimix hinglid --epochs 3

**Optional, bigger (~1 h): add 500k HingCorpus tweets.** Only run this *instead of* the `ourbert` cell above (delete `results/pretrained/ourbert` first if it exists). If `gdown` hits a Drive quota error: open the HingCorpus link in github.com/l3cube-pune/code-mixed-nlp → *Add shortcut to Drive* → pass `--hingcorpus_path /content/drive/MyDrive/<file>`.

In [ ]:
# !rm -rf results/pretrained/ourbert
# !python -m src.mlm_pretrain --name ourbert --sources sentimix hinglid hingcorpus --hingcorpus_lines 500000 --epochs 1

In [ ]:
# pre-training effect: MLM perplexity before -> after (corpus held-out, SentiMix val text)
import json
for n in ['ourbert-tapt','ourbert']:
    p = f'results/pretrained/{n}/pretrain_info.json'
    if os.path.exists(p):
        d = json.load(open(p)); print(n, d['corpus']['total_train_lines'], 'lines'); print(json.dumps(d['perplexity'], indent=1))

## 2. Fine-tune with the same protocol (LR sweep on val → 3 seeds on test)

In [ ]:
!python -m src.sweep --models ourbert-tapt --save_model

In [ ]:
!python -m src.sweep --models ourbert --save_model

In [ ]:
!python -m src.sweep --models hingmbert --save_model

## 3. Results

In [ ]:
!python -m src.analyze
print(open('results/tables/main_results.md').read())

In [ ]:
from IPython.display import Image, display
for f in ['main_results','f1_by_cmi','confusion_matrices']:
    display(Image(f'results/figures/{f}.png', width=750))